---
# NLI Probing
> Laura Komorek
---

This Notebook evaluates the NLI-based probing approach for our entity typing.

Our goal is to test if a pretrained Natural Language Inference model can
correctly determine if an entity belongs to a specified type.

For our project we reformulate entity typing as an NLI task:\
Premise: original sentence\
Hypotheses: "ENTITY is a TYPE"

For example:\
"premise": "EU rejects German call to boycott British lamb."\
"hypothesis": "EU is a per."\
(This example is taken from the OntoNotes dataset)

The model has to predict whether the hypothesis is entailed by the given premise.

We are evaluating this approach on our three datasets with different levels of label granularity:
- OntoNotes (coarse-grained)
- FIGER (medium-grained)
- Ultra-Fine Entity Typing (fine-grained)

Our model is the pratrained NLI model:\
RoBERTa-large-MNLI


---
## Imports

In [2]:
import json
import os
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from tqdm import tqdm
import pandas as pd
from sklearn.metrics import accuracy_score, classification_report

c:\Users\Laura\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


---

## Device Setup

If a GPU is available, it will be used automatically.
Otherwise the model will run on CPU.

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# to check which device is being used:
print("Device:", device)

Device: cpu


---

## Loading the NLI Model

As mentioned above, we are using the pretained model RoBERTa-large-MNLI.

However this model was trained on the MNLI dataset, therefore it predicts three classes:
- contradiction
- neutral
- entailment

For our datasets we have however only created a binary classification being **entailment** or **non entailment**.
So for our task we will convert the model's predictions into a binary decision as well:
- ENTAILMENT = entailment
- NOT ENTAILMENT = contradiction or neutral

In [5]:
model_name = "textattack/roberta-base-mnli"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)

model.to(device)
model.eval()

Some weights of the model checkpoint at textattack/roberta-base-mnli were not used when initializing RobertaForSequenceClassification: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


RobertaForSequenceClassification(
  (roberta): RobertaModel(
    (embeddings): RobertaEmbeddings(
      (word_embeddings): Embedding(50265, 768, padding_idx=1)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): RobertaEncoder(
      (layer): ModuleList(
        (0-11): 12 x RobertaLayer(
          (attention): RobertaAttention(
            (self): RobertaSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): RobertaSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
         

---

## NLI Probing 

The following function evaluates the NLI model on the given dataset.

It will follow following steps:
1. Load the given dataset containing the premise-hypothesis pairs
2. Run the NLI model on each of the pairs
3. Conver the model output to a binary prediction:
- entailment = ENTAILMENT
- contradiction or neutral = NOT ENTAILMENT
4. Store predictions together with the original data

The output is saved a a JSON file so that we can work with the evaluations metrics later on.

In [10]:
def run_nli_probing(input_file, output_file, batch_size=32):
    
    # to check which dataset is currently being loaded
    print(f"\nLoading Dataset: {input_file}")

    with open(input_file, encoding="utf-8") as f:
        data = json.load(f)
    
    results = []

    
    premises = [x["premise"] for x in data]
    hypotheses = [x["hypothesis"] for x in data]
    
    for i in tqdm(range(0, len(data), batch_size)):

        batch_premises = premises[i:i+batch_size]
        batch_hypotheses = hypotheses[i:i+batch_size]

        inputs = tokenizer(
            batch_premises,
            batch_hypotheses,
            padding=True,
            truncation=True,
            return_tensors="pt"
        ).to(device)

        with torch.no_grad():
            outputs = model(**inputs)

        preds = torch.argmax(outputs.logits, dim=1).cpu().tolist()

        for j, pred in enumerate(preds):
            if pred == 2:
                prediction = "ENTAILMENT"
            else:
                prediction = "NOT ENTAILMENT"

            results.append({
                "premise": batch_premises[j],
                "hypothesis": batch_hypotheses[j],
                "gold_label": data[i+j]["label"],
                "prediction": prediction
            })

    with open(output_file, "w") as f:
        json.dump(results, f, indent=2, ensure_ascii=False)

---

## Dataset Paths

Here we are defining the datasets that will be processd.

Each dataset already contains premise-hypothesis pairs that we created during the preprocessing.

In [16]:
datasets = {
    #"nli_ontonotes.json": "ontonotes_predictions.json",
    "nli_figer.json": "figer_predictions.json",
    #"nli_ultra.json": "ultrafine_predictions.json"
}

---

## Probing on all datasets

Here we run the NLI model on each dataset.

For each of the datasets, predictions are generated and saved as a separate JSON file.
These predictions files will also be used later on for evaluation.

In [14]:
for dataset, output in datasets.items():
    print("Currently running NLI probing for:", dataset)

    run_nli_probing(dataset,output)

Currently running NLI probing for: nli_ultra.json

Loading Dataset: nli_ultra.json


  0%|          | 35/471841 [00:39<147:16:11,  1.12s/it]


KeyboardInterrupt: 

---

## Final Output

Now after running this notebook, there will be three prediction files:
- ontonotes_predictions.json
- figer_predictions.json
- ultrafine_predictions.json

Each file contains:
- the original premise
- the hypothesis
- the gold label
- and finally the model's prediction